# Stone Age NLP · Core Architecture (core/)


This notebook is the core architecture of `stone-age-NLP` (V1 → V4).
Each section maps to an independent module under `core/`, and can be
extracted back to plain `.py` files with `extract_to_classic.py`.

SPDX-License-Identifier: CC-BY-4.0


## config — configuration


Global configuration and special tokens: vocabulary heads and the sentence separator.


In [ ]:
VOCAB_HEADS = ['<pad>', '<start>', '<end>', '<unk>']
SEP = '_'


## test — V1 word splitting + base statistics


Whitespace word splitting and base word statistics (bigrams + vocabulary).


In [ ]:
"""V1 core: word splitting and base word statistics."""

import re


bigram_counts = {}
all_tokens = set()
idx2word = {}
word2idx = {}


def split_words(text: str) -> list:
    """Split English text into words on whitespace.

    Sentence-ending punctuation (., !, ?, ;) is turned into the SEP token so
    that sentence boundaries are preserved in the statistics.
    """
    text = re.sub(r'[.!?;:]+', f' {SEP} ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.split()


def count_bigrams(tokens: list) -> None:
    """Count how often each word is immediately followed by another."""
    for i in range(1, len(tokens)):
        prev, nxt = tokens[i - 1], tokens[i]
        dic = bigram_counts.setdefault(prev, {})
        dic[nxt] = dic.get(nxt, 0) + 1


def update_vocab(new_words: list) -> None:
    """Extend the vocabulary with new tokens and rebuild the index maps."""
    all_tokens.update(new_words)
    vocab = VOCAB_HEADS + sorted(all_tokens)
    idx2word.clear()
    idx2word.update(dict(enumerate(vocab)))
    word2idx.clear()
    word2idx.update({w: i for i, w in idx2word.items()})


def load_from_file(filename: str) -> None:
    """Read a corpus file, split it into words and update statistics."""
    with open(filename, mode='r', encoding='utf-8', errors='ignore') as file:
        tokens = split_words(file.read())
    count_bigrams(tokens)
    update_vocab(tokens)



## test_func — prediction toolkit


Prediction utilities built on the adjacency dictionary, incl. BFS/DFS graph search.


In [ ]:
"""Prediction utilities built on the adjacency dictionary (incl. BFS/DFS graph search)."""

import random



def sort_by_weight(word_dict, word):
    """Sort the neighbours of a word by their counts, descending."""
    return sorted(word_dict[word].items(), key=lambda d: d[1], reverse=True)


def predict_next(word_dict, word):
    """Greedy prediction: always pick the most likely next word until SEP."""
    print(word, end='')
    while word != SEP:
        word = sort_by_weight(word_dict, word)[0][0]
        print(' ' + word, end='')
    print()


def predict_next_rand(word_dict, word):
    """Random prediction sampled from the weighted distribution until SEP."""
    print(word, end='')
    while word != SEP:
        choices = list(word_dict[word].keys())
        weights = list(word_dict[word].values())
        word = random.choices(choices, weights=weights, k=1)[0]
        print(' ' + word, end='')
    print()


def predict_next_rand_r(word_dict, word, limit=None):
    """Random prediction with a limit on the candidate window.

    ``limit`` can be an int (keep top-N), a (min, max) tuple (random slice),
    or a [start, stop] list (explicit slice).
    """
    print(word, end='')
    while True:
        lst = sort_by_weight(word_dict, word)
        if type(limit) is type(int()):
            lst = lst[:limit]
        if type(limit) is type(tuple()):
            lst = lst[:random.randint(*limit)]
        if type(limit) is type(list()):
            lst = lst[limit[0]:limit[1]]
        dic = dict(lst)
        new_word = random.choices(list(dic.keys()), weights=list(dic.values()), k=1)[0]
        if not new_word == SEP:
            word = new_word
            print(' ' + word, end='')
        else:
            if len(word_dict[word]) == 1:
                print(word_dict[word])
                break
            else:
                print('-', end='')


def predict_by_recursion(word_dict, start, end, limit):
    """Find a path from ``start`` to ``end`` via recursive DFS over the top-N candidates."""
    if start == end:
        return end
    if start == SEP:
        return ''
    for candidate, _ in sort_by_weight(word_dict, start)[:limit]:
        result = predict_by_recursion(word_dict, candidate, end, limit)
        if result and result != SEP:
            return start + result


def predict_by_loop(word_dict, start, end, limit, bfs: bool = False):
    """Expand a graph of nodes from ``start`` to ``end`` (BFS or DFS).

    Returns ``all_nodes``, a dict of word -> WNode used by ``get_path``.
    """
    class WNode:
        def __init__(self, data):
            self.data = data
            self.came_from = []
            self.next = []

    lst = [WNode(start)]
    all_nodes = dict()
    while lst:
        # BFS pops from the front, DFS pops from the back.
        cur_node = lst.pop(bfs - 1)
        if cur_node.data not in all_nodes:
            all_nodes[cur_node.data] = cur_node
        for wd, _count in sort_by_weight(word_dict, cur_node.data)[:limit]:
            if wd not in all_nodes:
                node = WNode(wd)
                node.came_from.append(cur_node)
                if not (cur_node.data == end or wd == SEP):
                    lst.append(node)
                    cur_node.next.append(node)
            else:
                node = all_nodes[wd]
                node.came_from.append(cur_node)
    return all_nodes


def get_path(all_nodes, start, end, direction):
    """Reconstruct a path from the graph built by ``predict_by_loop``.

    ``direction > 0`` walks forward through ``next``; ``direction < 0`` walks
    backward through ``came_from``.
    """
    path = []
    if direction > 0:
        cur_node = all_nodes[start]
        while True:
            path.append(cur_node.data)
            if cur_node == all_nodes[end]:
                return path
            cur_node = cur_node.next[direction - 1]
    if direction < 0:
        cur_node = all_nodes[end]
        while True:
            path.append(cur_node.data)
            if cur_node == all_nodes[start]:
                path.reverse()
                return path
            cur_node = cur_node.came_from[-direction - 1]



## test2 — V2 co-occurrence + weighted prediction


Pairwise co-occurrence statistics + weighted prediction: `0.5×adjacency + 0.25×co-occurrence`.


In [ ]:
"""V2: co-occurrence statistics + weighted prediction."""


cooccurrence_counts = {}


def sort_desc(dic):
    """Sort a {key: count} dict by count, descending."""
    return sorted(dic.items(), key=lambda d: d[1], reverse=True)


def count_cooccurrence(tokens: list) -> None:
    """Count how often two words appear in the same sentence."""
    sentence = []
    for word in tokens:
        if word != SEP:
            sentence.append(word)
        else:
            for cur_word in sentence:
                dic = cooccurrence_counts.setdefault(cur_word, {})
                for other in sentence:
                    if other != cur_word and other != SEP:
                        dic[other] = dic.get(other, 0) + 1
            sentence = []


def weighted_score(input_words, bigram_counts, cooccurrence_counts):
    """V2 score: 0.5 * adjacency + 0.25 * co-occurrence."""
    adj_bonus, cooc_bonus, candidates = {}, {}, {}
    for word in input_words:
        for cand, count in cooccurrence_counts.get(word, {}).items():
            cooc_bonus[cand] = cooc_bonus.get(cand, 0) + count
            candidates.setdefault(cand, 0)
        for cand, count in bigram_counts.get(word, {}).items():
            adj_bonus[cand] = adj_bonus.get(cand, 0) + count
            candidates.setdefault(cand, 0)
    for cand in candidates:
        candidates[cand] = adj_bonus.get(cand, 0) * 0.5 + cooc_bonus.get(cand, 0) * 0.25
    ranked = sort_desc(candidates)
    return ranked[:len(ranked) // 2]


def generate(words, ignore_words=''):
    """Predict a continuation from the given words and print it, until SEP."""
    lst = list(words)
    print(' '.join(lst), end='')
    while lst[-1] != SEP:
        ranked = weighted_score(lst, bigram_counts, cooccurrence_counts)
        while ranked[0][0] in ignore_words:
            ranked.pop(0)
        new_word = ranked[0][0]
        lst.append(new_word)
        print(' ' + new_word, end='')
    print()
    return lst



## test3 — V3 distance-weighted prediction


Adds distance weights to the V2 score: `0.5×adjacency + 0.25×co-occurrence + 0.25×(1/min distance)`.


In [ ]:
"""V3: distance-weighted prediction."""


distance_stats = {}


def count_distances(tokens: list) -> None:
    """Record the min/max in-sentence distance between each pair of words."""
    sentence = []
    for word in tokens:
        if word != SEP:
            sentence.append(word)
        else:
            for cur_word in sentence:
                dic = distance_stats.setdefault(cur_word, {})
                for other in sentence:
                    if other != cur_word and other != SEP:
                        dic.setdefault(other, [float('inf'), 0])
                        dist = abs(sentence.index(other) - sentence.index(cur_word))
                        if dist < dic[other][0]:
                            dic[other][0] = dist
                        if dist > dic[other][1]:
                            dic[other][1] = dist
            sentence = []


def minmax_normalize(stats, depth) -> None:
    """Normalize each dimension to [0, 1] across all candidates."""
    for i in range(1, depth):
        values = [v[i] for v in stats.values()]
        lo, hi = min(values), max(values)
        if lo == hi:
            continue
        for v in stats.values():
            v[i] = (v[i] - lo) / (hi - lo)


def weighted_score_v3(input_words, bigram_counts, cooccurrence_counts, distance_stats):
    """V3 score: 0.5*adjacency + 0.25*co-occurrence + 0.25*(1/min distance)."""
    points = [0, 0.5, 0.25, 0.25]
    scores = {}
    for word in input_words:
        for cand, count in bigram_counts.get(word, {}).items():
            scores.setdefault(cand, [0, 0, 0, 0])[1] += count
        for cand, count in cooccurrence_counts.get(word, {}).items():
            scores.setdefault(cand, [0, 0, 0, 0])[2] += count
        for cand, dist in distance_stats.get(word, {}).items():
            scores.setdefault(cand, [0, 0, 0, 0])[3] += 1 / dist[0]
    minmax_normalize(scores, 3)
    for v in scores.values():
        v[0] = sum(v[i] * points[i] for i in range(1, 4))
    ranked = sorted(scores.items(), key=lambda d: d[1][0], reverse=True)
    return ranked[:len(ranked) // 2]


def generate_v3(words, ignore_words=''):
    """Predict a continuation using V3 distance weights, avoiding repeats."""
    lst = list(words)
    print(' '.join(lst), end='')
    while lst[-1] != SEP:
        ranked = weighted_score_v3(lst, bigram_counts, cooccurrence_counts, distance_stats)
        while ranked[0][0] in ignore_words or ranked[0][0] in lst:
            ranked.pop(0)
        new_word = ranked[0][0]
        lst.append(new_word)
        print(' ' + new_word, end='')
    print()
    return lst



## test4 — V4 graph dictionary tree


Graph-structure dictionary tree with nodes, caches and pluggable builders.


In [ ]:
"""V4: graph structure dictionary tree (nodes + caches + custom builders)."""



class WordConfig:
    """Configuration controlling the graph behaviour of ``WordNode``."""

    def __init__(self,
                 fast_mode=1,             # cache mode: 0 none, 1 lazy, 2 full
                 mode='X',                # 'B' undirected (bidirectional), 'X' directed (mixed)
                 builder=None,            # function dict -> any, dict in _links format
                 weight_aggregator=None): # function weights_lst -> number
        """
        :param mode: graph mode, 'B' undirected (bidirectional), 'X' directed (default).
        """
        self.fast_mode = fast_mode
        self.mode = mode
        self.builder = builder or self.build_origin_dict
        self.weight_aggregator = weight_aggregator or (lambda lst: sum(lst))

    # -------- preset builders --------
    def build_word_set(self, dic):
        """Return the set of neighbour words."""
        return {k.word for k in dic}

    def build_origin_dict(self, dic):
        """Return the raw format {node: edge info} without conversion."""
        return dic

    def build_list_weight(self, dic):
        """Return [(node, aggregated weight), ...]."""
        return [(k, self.weight_aggregator(v['weights_lst'])) for k, v in dic.items()]

    def build_dict_weight(self, dic):
        """Return {word: aggregated weight}."""
        return {k.word: self.weight_aggregator(v['weights_lst']) for k, v in dic.items()}

    def build_list_full(self, dic):
        """Return [(word, weights_list, other, direction), ...]."""
        return [(k.word, v['weights_lst'], v.get('other', {}), v.get('direct', '')) for k, v in dic.items()]

    def build_dict_full(self, dic):
        """Return {word: {'weights': [...], 'other': {...}, 'direct': '...'}}."""
        return {k.word: {
            'weights': v['weights_lst'],
            'other': v.get('other', {}),
            'direct': v.get('direct', '')
        } for k, v in dic.items()}

    # -------- preset templates for end users --------
    @classmethod
    def words(cls, fast_mode=2, mode='X'):
        """Config that builds a set of words."""
        cfg = cls(fast_mode=fast_mode, mode=mode)
        cfg.builder = cfg.build_word_set
        return cfg

    @classmethod
    def default(cls, fast_mode=1, mode='X', agg=None):
        """Config that builds a [(node, aggregated weight), ...] list."""
        cfg = cls(fast_mode=fast_mode, mode=mode, weight_aggregator=agg)
        cfg.builder = cfg.build_list_weight
        return cfg

    @classmethod
    def dict_full(cls, fast_mode=0, mode='X'):
        """Config that builds {word: {"weights": [...], "other": {...}, "direct": "..."}}."""
        cfg = cls(fast_mode=fast_mode, mode=mode)
        cfg.builder = cfg.build_dict_full
        return cfg


DEFAULT_CONFIG = WordConfig.default()


class WordNode:
    def __init__(self, word, config=None):
        self.word = word
        self.config = config or DEFAULT_CONFIG

        # Single source of truth: neighbour node -> edge info
        self._links = {}  # {node_obj: {"weights_lst": [], "other": {}, "direct": ""}}

        if int(self.config.fast_mode) > 0:
            self.parents_lst = None
            self.children_lst = None

    def _get_links(self, direct=''):
        if direct == '':
            direct = 'pc'
        selected = {}
        for k, v in self._links.items():
            d = v.get('direct', '')
            if self.config.mode == 'B' or d in direct:
                selected[k] = v
        return self.config.builder(selected)

    def get_links(self, direct=''):
        if direct == '':
            direct = 'pc'
        fast_mode = int(self.config.fast_mode)
        if fast_mode == 0:
            return self._get_links(direct)
        if fast_mode > 0:
            if (direct in ('p', 'pc') and self.parents_lst is None) or \
               (self.config.mode == 'B' or direct in ('c', 'pc') and self.children_lst is None):
                self.update_cache()
            if self.config.mode == 'B' or direct == 'c':
                return self.children_lst
            if direct == 'p':
                return self.parents_lst
            if direct == 'pc':
                return self._merge(self.parents_lst, self.children_lst)

    def add_links(self, node, weights_lst, direct='c'):
        edge = self._links.setdefault(node, {'weights_lst': [], 'other': {}, 'direct': ''})
        edge['weights_lst'] = weights_lst
        if self.config.mode != 'B':
            edge['direct'] = self._merge_direct(edge['direct'], direct)
        if self.config.fast_mode == 2 or self.parents_lst is not None or self.children_lst is not None:
            self.update_cache()
        return edge

    def update_links(self, node, weights_lst=None, direct=None, other=None):
        edge = self._links.get(node)
        if edge is None:
            return None
        if weights_lst:
            edge['weights_lst'] = weights_lst
        if direct:
            edge['direct'] = self._merge_direct(edge['direct'], direct)
        if other:
            edge['other'].update(other)
        if self.parents_lst is not None or self.children_lst is not None:
            self.update_cache()
        return edge

    def update_cache(self):
        if int(self.config.fast_mode) == 0:
            return
        if self.config.mode == 'B':
            self.children_lst = self._get_links('c')
        else:
            self.parents_lst = self._get_links('p')
            self.children_lst = self._get_links('c')

    def _merge(self, a, b):
        if isinstance(a, set):
            return a | b
        if isinstance(a, dict):
            return {**a, **b}
        if isinstance(a, list):
            return a + b
        return (a, b)

    @staticmethod
    def _merge_direct(d0, d1):
        if not d0:
            return d1
        if not d1:
            return d0
        return d0 if d0 == d1 else 'pc'


from itertools import pairwise

word_index_dic = dict()


def build_dictionary_tree_sample(cut_res, config=None):
    """Build a sample dictionary tree from adjacent token pairs."""
    tmp = {}
    for prev, nxt in pairwise(cut_res):
        if SEP not in (prev, nxt):
            tmp.setdefault(prev, {})
            tmp[prev][nxt] = tmp[prev].get(nxt, 0) + 1
    for word, children in tmp.items():
        node = word_index_dic.setdefault(word, WordNode(word, config))
        for child_word, count in children.items():
            child_node = word_index_dic.setdefault(child_word, WordNode(child_word, config))
            node.add_links(child_node, [count], direct='c')

